<a href="https://colab.research.google.com/github/vidhi-sys/River_Discharge_LSTM/blob/main/DL_WATER_QUALITY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
import pandas as pd
data=pd.read_csv('/content/riverdischarge_manual_daily_madhya-pradesh-sw_mp_1950_2000.csv')
data.head(5)
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler


,SlNo,Station,Agency,State LGD Code,State,District LGD Code,District,Tehsil,Block,Village,River,Basin,Tributary,Subtributary,SubSubtributary,Local River,Latitude,Longitude,Data Acquisition Time,Manual Daily River Water Discharge (m3/sec)
0,1,Anterbeliya,Madhya Pradesh SW,23,Madhya Pradesh,412,JHABUA,JHABUA,-,-,Mahi,Mahi,Anas,-,-,Anas,22.857778,74.551389,01-01-2000 08:30,0.0
1,2,Anterbeliya,Madhya Pradesh SW,23,Madhya Pradesh,412,JHABUA,JHABUA,-,-,Mahi,Mahi,Anas,-,-,Anas,22.857778,74.551389,02-01-2000 08:30,0.0
2,3,Anterbeliya,Madhya Pradesh SW,23,Madhya Pradesh,412,JHABUA,JHABUA,-,-,Mahi,Mahi,Anas,-,-,Anas,22.857778,74.551389,03-01-2000 08:30,0.0
3,4,Anterbeliya,Madhya Pradesh SW,23,Madhya Pradesh,412,JHABUA,JHABUA,-,-,Mahi,Mahi,Anas,-,-,Anas,22.857778,74.551389,04-01-2000 08:30,0.0
4,5,Anterbeliya,Madhya Pradesh SW,23,Madhya Pradesh,412,JHABUA,JHABUA,-,-,Mahi,Mahi,Anas,-,-,Anas,22.857778,74.551389,05-01-2000 08:30,0.0


# **DATA-CLEANING**

In [42]:
data=data.drop(['State'],axis=1)
data=data.drop(['Block'],axis=1)
data=data.drop(['Village'],axis=1)
data=data.drop(['Subtributary'],axis=1)

data=data.drop(['State LGD Code'],axis=1)


In [43]:
data=data.drop(['SubSubtributary'],axis=1)
data=data.drop(['Agency'],axis=1)
data.head(5)


,SlNo,Station,District LGD Code,District,Tehsil,River,Basin,Tributary,Local River,Latitude,Longitude,Data Acquisition Time,Manual Daily River Water Discharge (m3/sec)
0,1,Anterbeliya,412,JHABUA,JHABUA,Mahi,Mahi,Anas,Anas,22.857778,74.551389,01-01-2000 08:30,0.0
1,2,Anterbeliya,412,JHABUA,JHABUA,Mahi,Mahi,Anas,Anas,22.857778,74.551389,02-01-2000 08:30,0.0
2,3,Anterbeliya,412,JHABUA,JHABUA,Mahi,Mahi,Anas,Anas,22.857778,74.551389,03-01-2000 08:30,0.0
3,4,Anterbeliya,412,JHABUA,JHABUA,Mahi,Mahi,Anas,Anas,22.857778,74.551389,04-01-2000 08:30,0.0
4,5,Anterbeliya,412,JHABUA,JHABUA,Mahi,Mahi,Anas,Anas,22.857778,74.551389,05-01-2000 08:30,0.0


In [44]:
data.tail(5)

,SlNo,Station,District LGD Code,District,Tehsil,River,Basin,Tributary,Local River,Latitude,Longitude,Data Acquisition Time,Manual Daily River Water Discharge (m3/sec)
4707,4708,Ugli,428,SEONI,KEOLARI,Godavari,Godavari,Wainganga,Wainganga,22.238611,80.0875,27-12-2000 08:00,11.6412
4708,4709,Ugli,428,SEONI,KEOLARI,Godavari,Godavari,Wainganga,Wainganga,22.238611,80.0875,28-12-2000 08:00,11.2964
4709,4710,Ugli,428,SEONI,KEOLARI,Godavari,Godavari,Wainganga,Wainganga,22.238611,80.0875,29-12-2000 08:00,11.2100
4710,4711,Ugli,428,SEONI,KEOLARI,Godavari,Godavari,Wainganga,Wainganga,22.238611,80.0875,30-12-2000 08:00,11.2334
4711,4712,Ugli,428,SEONI,KEOLARI,Godavari,Godavari,Wainganga,Wainganga,22.238611,80.0875,31-12-2000 08:00,11.2194


In [45]:
imp_data = data[['Data Acquisition Time', 'Manual Daily River Water Discharge (m3/sec)','Station','River']].copy()
imp_data['Timestamp'] = pd.to_datetime(imp_data['Data Acquisition Time'], format='%d-%m-%Y %H:%M')
imp_data['Month'] = imp_data['Timestamp'].dt.month
imp_data['Time'] = imp_data['Timestamp'].dt.time
imp_data = imp_data[['Manual Daily River Water Discharge (m3/sec)', 'Month', 'Time', 'Station', 'River']]

In [46]:
imp_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4712 entries, 0 to 4711
Data columns (total 5 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Manual Daily River Water Discharge (m3/sec)  4712 non-null   float64
 1   Month                                        4712 non-null   int32  
 2   Time                                         4712 non-null   object 
 3   Station                                      4712 non-null   object 
 4   River                                        4712 non-null   object 
dtypes: float64(1), int32(1), object(3)
memory usage: 165.8+ KB


In [47]:
imp_data.head()

,Manual Daily River Water Discharge (m3/sec),Month,Time,Station,River
0,0.0,1,08:30:00,Anterbeliya,Mahi
1,0.0,1,08:30:00,Anterbeliya,Mahi
2,0.0,1,08:30:00,Anterbeliya,Mahi
3,0.0,1,08:30:00,Anterbeliya,Mahi
4,0.0,1,08:30:00,Anterbeliya,Mahi


In [48]:
imp_data = imp_data.rename(columns={
    'Manual Daily River Water Discharge (m3/sec)': 'Discharge'
})

In [49]:
imp_data.head()


,Discharge,Month,Time,Station,River
0,0.0,1,08:30:00,Anterbeliya,Mahi
1,0.0,1,08:30:00,Anterbeliya,Mahi
2,0.0,1,08:30:00,Anterbeliya,Mahi
3,0.0,1,08:30:00,Anterbeliya,Mahi
4,0.0,1,08:30:00,Anterbeliya,Mahi


In [50]:
imp_data = imp_data.sort_values(['Station','Month', 'Time'])
imp_data['Discharge'].describe()

,Discharge
count,4712.000000
mean,12.909286
std,48.743165
min,0.000000
25%,0.000000
50%,0.372975
75%,6.892152
max,1587.210000


In [51]:
import numpy as np
imp_data['Discharge'].replace(0, np.nan, inplace=True)
imp_data['Discharge'].interpolate(inplace=True)

/tmp/ipython-input-768/3089474399.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  imp_data['Discharge'].replace(0, np.nan, inplace=True)
/tmp/ipython-input-768/3089474399.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=

# **DEEP LEARNING MODEL**

In [52]:
imp_data['Station'].unique()

array(['Anterbeliya', 'Bamnia', 'Bandol', 'Bonkatta', 'Chand',
       'Chhindlai', 'Dabri', 'Deoghat', 'Hirankheri', 'KUNDANPUR',
       'Kiranapur', 'MAHI', 'Manpur', 'Mohuda', 'Palinga', 'Pansemal',
       'Rambhapur', 'Rampaili', 'Ratanpur', 'Teska', 'Ugli'], dtype=object)

Choosing Station = **Anterbeliya**

In [53]:
station_data = imp_data[imp_data['Station'] == 'Anterbeliya'].copy()
station_data = station_data.sort_values('Time')

In [54]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

scaler = MinMaxScaler()
scaled = scaler.fit_transform(station_data[['Discharge']])

In [55]:
station_data['lag1'] = station_data['Discharge'].shift(1)
station_data['lag2'] = station_data['Discharge'].shift(2)
station_data['lag3'] = station_data['Discharge'].shift(3)

station_data = station_data.dropna()

In [56]:
def create_sequences(data, seq_len=7):
    X, y = [], []
    for i in range(len(data)-seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled, 7)

In [60]:
from sklearn.ensemble import RandomForestRegressor

features = ['Month']
training_data = station_data.dropna(subset=['Discharge'])

X_rf = training_data[features]
y_rf = training_data['Discharge']

rf = RandomForestRegressor()
rf.fit(X_rf, y_rf)

RandomForestRegressor()

In [61]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(7,1)),
    LSTM(32),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.fit(X, y, epochs=30, batch_size=16)

Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - loss: nan
Epoch 2/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: nan
Epoch 3/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: nan
Epoch 4/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: nan
Epoch 5/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: nan
Epoch 6/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: nan
Epoch 7/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: nan
Epoch 8/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: nan
Epoch 9/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: nan
Epoch 10/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: nan
Epoch 11/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: nan
Epoch 12/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: nan
Epoch 13/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: nan
Epoch 14/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: nan
Epoch 15/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: nan
Epoch 16/30
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: nan
Epoch 

In [62]:
def classify_flood(x):
    if x < 200:
        return 0  # Low
    elif x < 500:
        return 1  # Moderate
    else:
        return 2  # High

station_data['Flood_Risk'] = station_data['Discharge'].apply(classify_flood)

In [63]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier()
clf.fit(X_rf, station_data['Flood_Risk'])

RandomForestClassifier()

# **SAVING THE MODEL**

In [64]:
model.save("river_lstm_model.keras")

In [65]:
!pip install streamlit
import streamlit as st

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 94.9 MB/s eta 0:00:00


In [70]:
import streamlit as st
import pandas as pd

st.title("River Discharge & Flood Risk System")

sample_month = 6
sample_input = pd.DataFrame({'Month': [sample_month]})

prediction = rf.predict(sample_input)[0]

if prediction < 200:
    st.success("Low Flood Risk")
elif prediction < 500:
    st.warning("Moderate Flood Risk")
else:
    st.error("High Flood Risk")
st.line_chart(station_data.set_index('Time')['Discharge'])

2026-03-02 04:11:04.709 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-02 04:11:04.712 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-02 04:11:04.714 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-02 04:11:04.761 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-02 04:11:04.765 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-02 04:11:04.769 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-02 04:11:10.341 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-02 04:11:10.342 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()